# Tears of Steel: script Knowledge Base (optional)

This notebook adds the film's script to the deployment's Bedrock Knowledge Base, so scene answers can draw on it. It follows the v1 sample's [`script_analysis.ipynb`](https://github.com/aws-samples/sample-hyper-personalized-agentic-ai-media-assistant/blob/v1/backend/tools/script_analysis.ipynb), but uses the Knowledge Base and source bucket the Application stack already created.

The scene analysis from `1-video-scenes.ipynb` works without it.

**Get the script.** It is not in this repository. Download it from [Blender Studio](https://studio.blender.org/projects/tears-of-steel/55f344892beb3300251b0172/?asset=34) (free registration required) and set `SCRIPT_PATH` below. A PDF is converted to text with Bedrock Data Automation; a plain-text file is used as is.

Run it with the same credentials as the first notebook (Bedrock Data Automation, S3 and `bedrock:StartIngestionJob` on the Knowledge Base).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import tears_of_steel as tos

CONFIG = "../../infra/environments/<name>.json"   # your environment file
AWS_PROFILE = None
SCRIPT_PATH = "<path to the downloaded script>"   # .pdf or .txt
WORKDIR = Path("work")

dep = tos.Deployment(CONFIG, profile=AWS_PROFILE)
script = Path(SCRIPT_PATH)

## 1. Extract the text
A PDF goes through a BDA document project (plain text and markdown output, as in v1); a text file is read directly.

In [ ]:
if script.suffix.lower() == ".pdf":
    project_arn = dep.bda_project(f"{dep.prefix}-tears-of-steel-script", {
        "document": {
            "extraction": {"granularity": {"types": ["DOCUMENT", "PAGE", "ELEMENT"]}, "boundingBox": {"state": "DISABLED"}},
            "generativeField": {"state": "DISABLED"},
            "outputFormat": {"textFormat": {"types": ["PLAIN_TEXT", "MARKDOWN"]}, "additionalFileFormat": {"state": "DISABLED"}},
        }
    })
    key = f"{tos.BDA_PREFIX}/script/{script.name}"
    dep.upload(script, dep.media_bucket, key)
    result = dep.run_bda(project_arn, input_uri=f"s3://{dep.media_bucket}/{key}",
                         output_uri=f"s3://{dep.media_bucket}/{tos.BDA_PREFIX}/script-output")
    text = result["document"]["representation"]["text"]
else:
    text = script.read_text(encoding="utf-8")
cleaned = tos.clean_script(text)
print(len(cleaned.splitlines()), "lines")
print(cleaned[:800])

## 2. Upload to the Knowledge Base source
The cleaned script and a metadata file go to the Application stack's `KnowledgeSourceBucket` under `scriptsPrefix`. The metadata tags the script with `video_name`, which the agent filters on.

In [ ]:
import json
WORKDIR.mkdir(exist_ok=True)
txt = WORKDIR / tos.SCRIPT_NAME
txt.write_text(cleaned, encoding="utf-8")
meta = WORKDIR / f"{tos.SCRIPT_NAME}.metadata.json"
meta.write_text(json.dumps(tos.kb_metadata()))
prefix = dep.cfg.get("scriptsPrefix", "input/kb/")
bucket = dep.outputs["KnowledgeSourceBucket"]
dep.upload(txt, bucket, prefix + txt.name, content_type="text/plain")
dep.upload(meta, bucket, prefix + meta.name, content_type="application/json")

## 3. Sync the Knowledge Base
Start an ingestion job on the stack's data source and wait for it.

In [ ]:
dep.ingest_knowledge_base()

## Done
Scene answers for *Tears of Steel* now also use the script.